# Humanization — pilot di 42 fonti (PDF §§1.6–1.7)

Notebook di riproduzione di riscritture, detector, stile e conservazione semantica. Il caricamento utilizza i tre CSV ufficiali inclusi nel repository; non scarica copie da mirror.

Abilitare GPU e Internet. Iniziare con `MODE="smoke"` (3 fonti), poi `MODE="pilot"` (42). Per i comandi e i due preset del pilot leggere [README_Pilot.md](README_Pilot.md).


## 1. Configurazione e controllo della GPU
Cambia soltanto MODE quando passi dalla prova tecnica al pilot. Mantieni un RUN_SUFFIX nuovo se cambi sistemi, prompt o configurazioni. I 300 umani di calibrazione restano separati dalle fonti di test, anche dalle 126 fonti riservate per lo studio successivo.


In [ ]:
from pathlib import Path
import gc, hashlib, json, os, shutil, subprocess, sys, tempfile

MODE = "smoke"                     # "smoke" oppure "pilot"
RUN_SUFFIX = "_thesis_pilot"
ACTIVE_SYSTEMS = ["identity", "qwen_humanize"]
REUSE_CACHED_MODELS = None          # None: automatico; True: solo checkpoint locali completi
REPO_REF = "main"                  # si registra il commit effettivamente risolto
assert MODE in {"smoke", "pilot"}
assert Path("/kaggle/working").exists(), "Questo notebook è preparato per Kaggle."
TMP = Path("/kaggle/temp/humanization_runtime")  # vecchi asset: possono essere di sola lettura
RUN = Path("/kaggle/working") / ("humanization_" + MODE + RUN_SUFFIX)
RUNTIME_CACHE = RUN.parent / ".humanization_runtime_cache"
REPO = RUN.parent / ".humanization_code" / ("Humanization_Thesis_" + MODE + RUN_SUFFIX)

def writable_directory(directory):
    try:
        directory.mkdir(parents=True, exist_ok=True)
        probe = directory / ".humanization_write_probe"
        probe.write_text("ok")
        probe.unlink()
        return True
    except OSError:
        return False

for directory in (RUN, RUN / "_tmp", RUNTIME_CACHE, REPO.parent):
    assert writable_directory(directory), f"Directory non scrivibile: {directory}"
old_hub = TMP / "huggingface" / "hub"
old_runtime_writable = writable_directory(TMP)
if REUSE_CACHED_MODELS is None:
    REUSE_CACHED_MODELS = (not old_runtime_writable
                           and (TMP / "venv/bin/python").is_file()
                           and old_hub.is_dir() and any(old_hub.glob("models--*")))
if REUSE_CACHED_MODELS:
    assert old_hub.is_dir(), "Cache locale assente: usare una sessione scrivibile con download consentiti."
    hub_cache = old_hub
else:
    candidates = [old_hub, Path("/kaggle/temp/humanization_downloads_http/hub"),
                  Path("/tmp/humanization_downloads_http/hub"), RUNTIME_CACHE / "huggingface/hub"]
    hub_cache = next((p for p in candidates if writable_directory(p)), None)
    assert hub_cache is not None, "Nessuna cache Hugging Face scrivibile."

for key, value in {
    "HF_HOME": str(hub_cache.parent), "HF_HUB_CACHE": str(hub_cache),
    "HUMANIZATION_HF_READ_CACHE": str(old_hub),
    "HUMANIZATION_REUSE_CACHED_MODELS": "1" if REUSE_CACHED_MODELS else "0",
    "HF_HUB_OFFLINE": "1" if REUSE_CACHED_MODELS else "0",
    "TRANSFORMERS_OFFLINE": "1" if REUSE_CACHED_MODELS else "0",
    "HF_XET_CACHE": str(RUNTIME_CACHE / "xet"), "HF_HUB_DISABLE_XET": "1",
    "TMPDIR": str(RUN / "_tmp"), "TMP": str(RUN / "_tmp"), "TEMP": str(RUN / "_tmp"),
    "NLTK_DATA": os.pathsep.join(map(str, [RUNTIME_CACHE / "nltk_data", TMP / "nltk_data"])),
    "HUMANIZATION_NLTK_WRITE_DIR": str(RUNTIME_CACHE / "nltk_data"),
    "MPLCONFIGDIR": str(RUNTIME_CACHE / "matplotlib"), "XDG_CACHE_HOME": str(RUNTIME_CACHE / "cache"),
    "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "2",
    "USE_TF": "0", "USE_FLAX": "0", "USE_TORCH": "1", "HF_HUB_DOWNLOAD_TIMEOUT": "120",
}.items():
    os.environ[key] = value
for variable in ("TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE"):
    if variable in os.environ:
        os.environ[variable] = str(hub_cache)
tempfile.tempdir = None

import torch
from packaging.version import Version
assert torch.cuda.is_available(), "Abilita una GPU nelle impostazioni Kaggle."
assert Version(torch.__version__.split("+")[0]) >= Version("2.6"), "Serve PyTorch CUDA >= 2.6 per RADAR."
print("PyTorch:", torch.__version__)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Cache HF:", hub_cache, "— riuso locale:", REUSE_CACHED_MODELS)
print("Spazio libero cache HF (GiB):", round(shutil.disk_usage(hub_cache).free / 2**30, 1))
print("Clone e risultati scrivibili:", REPO, RUN)

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/FrancescoM00/Humanization-Thesis.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
previous = RUN / "runtime.json"
if previous.exists():
    assert json.loads(previous.read_text())["repository_commit"] == COMMIT, "Il codice è cambiato: scegli un RUN_SUFFIX nuovo."
print("Repository commit:", COMMIT)

# Import del codice condiviso anche se si riusa una venv non scrivibile.
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")


## 2. Dipendenze isolate e risorse linguistiche
La venv riusa PyTorch CUDA dell'immagine Kaggle. Il lockfile CPU del vecchio ambiente non viene installato. Il primo avvio scarica dipendenze, modello spaCy e CMUdict. La cache temporanea va riscaricata dopo la chiusura della sessione; non contiene risultati da conservare.


In [ ]:
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"

requirements = REPO / "requirements/kaggle.txt"
expected = hashlib.sha256(requirements.read_bytes()).hexdigest()
VENV = None
# A valid, matching old venv is readable even when its filesystem is read-only.
for candidate in (TMP / "venv", RUNTIME_CACHE / "venv"):
    python = candidate / "bin/python"
    stamp = candidate / "requirements.sha256"
    if python.is_file() and stamp.is_file() and stamp.read_text() == expected:
        probe = subprocess.run([str(python), "-m", "pip", "--version"], capture_output=True, text=True)
        if probe.returncode == 0:
            VENV = candidate
            break
if VENV is not None:
    print("Riutilizzo venv senza modificarla:", VENV)
else:
    base = TMP if old_runtime_writable else RUNTIME_CACHE
    VENV = base / "venv"
    assert writable_directory(VENV), f"Venv non scrivibile: {VENV}"
    python = VENV / "bin/python"
    # Kaggle may lack ensurepip; base pip manages this CUDA-preserving venv.
    subprocess.run([sys.executable, "-m", "venv", "--without-pip",
                    "--system-site-packages", str(VENV)], check=True)
    pip_probe = subprocess.run([str(python), "-m", "pip", "--version"], capture_output=True, text=True)
    if not (VENV / "bin/pip").is_file() or pip_probe.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "--python", str(python),
                        "install", "--upgrade", "--ignore-installed", "pip"], check=True)
    subprocess.run([str(python), "-m", "pip", "install", "-r", str(requirements)], check=True)
    (VENV / "requirements.sha256").write_text(expected)
PYTHON = VENV / "bin/python"
nltk_probe = """import os, nltk
try:
    nltk.data.find('corpora/cmudict')
    print('cmudict già disponibile')
except LookupError:
    nltk.download('cmudict', download_dir=os.environ['HUMANIZATION_NLTK_WRITE_DIR'], raise_on_error=True)
"""
subprocess.run([str(PYTHON), "-c", nltk_probe], check=True)

# Check concrete model imports in a fresh process, without downloading weights.
runtime_probe = 'import os\nos.environ["USE_TF"] = "0"\nos.environ["USE_FLAX"] = "0"\nos.environ["USE_TORCH"] = "1"\nimport torch, spacy, textstat\nfrom transformers import (Qwen2ForCausalLM, RobertaForSequenceClassification,\n                          DebertaV2ForSequenceClassification, T5ForConditionalGeneration)\nfrom sentence_transformers import SentenceTransformer\nfrom bert_score import BERTScorer\nnlp = spacy.load("en_core_web_sm")\nassert len(nlp("This is a short English passage.")) > 0\nprint("Import e modello spaCy: OK")\nassert torch.cuda.is_available(), "CUDA non disponibile nella venv"\nx = torch.ones((16, 16), device="cuda", dtype=torch.float16)\ny = x @ x\ntorch.cuda.synchronize()\nassert bool(torch.isfinite(y).all()), "Il controllo CUDA ha prodotto valori non finiti"\nprint("CUDA nella venv:", torch.__version__)\nprint("Kernel CUDA FP16: OK")\n'
subprocess.run([str(PYTHON), "-c", runtime_probe], check=True)

# Check the installed CUDA quantization backend before a large DIPPER download.
if any(s["name"] in ACTIVE_SYSTEMS and s.get("load_in_4bit")
       for s in json.loads((REPO / "configs/pilot.json").read_text())["humanizers"]):
    quantization_probe = 'import torch\nimport bitsandbytes as bnb\nassert torch.cuda.is_available(), "Abilitare una GPU Kaggle"\nlayer = bnb.nn.Linear4bit(64, 64, bias=False,\n                         compute_dtype=torch.float16,\n                         compress_statistics=True,\n                         quant_type="nf4").to("cuda:0").eval()\nx = torch.ones((1, 64), device="cuda:0", dtype=torch.float16)\nwith torch.inference_mode():\n    y = layer(x)\ntorch.cuda.synchronize()\nassert bool(torch.isfinite(y).all()), "Il controllo NF4 ha prodotto valori non finiti"\nprint("Backend CUDA 4-bit: OK")\n'
    subprocess.run([str(PYTHON), "-c", quantization_probe], check=True)

def run(script, *arguments):
    scratch = RUN / "_tmp"
    scratch.mkdir(parents=True, exist_ok=True)
    child_env = dict(os.environ)
    child_env.update({name: str(scratch) for name in ("TMPDIR", "TMP", "TEMP")})
    subprocess.run([str(PYTHON), str(REPO / "scripts" / script), *map(str, arguments)],
                   cwd=REPO, env=child_env, check=True)

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

CONFIG = RUN / "config.json"
config_source = REPO / "configs/pilot.json"
if CONFIG.exists():
    assert CONFIG.read_bytes() == config_source.read_bytes(), "Config cambiata: scegli un RUN_SUFFIX nuovo."
else:
    shutil.copy2(config_source, CONFIG)
config = json.loads(CONFIG.read_text())
runtime = {"repository_commit": COMMIT, "mode": MODE, "active_systems": ACTIVE_SYSTEMS,
           "torch": torch.__version__, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
           "requirements_sha256": expected,
           "venv_path": str(VENV), "hf_hub_cache": str(hub_cache),
           "reuse_cached_models": REUSE_CACHED_MODELS,
           "config_sha256": hashlib.sha256(CONFIG.read_bytes()).hexdigest()}
if previous.exists():
    assert json.loads(previous.read_text())["active_systems"] == ACTIVE_SYSTEMS, "Sistemi cambiati: scegli un RUN_SUFFIX nuovo."
previous.write_text(json.dumps(runtime, indent=2))
(RUN / "pip_freeze.txt").write_text(subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True))
print("Configurazione salvata:", CONFIG)


## 3. CSV ufficiali e selezione congelata

I CSV originali vengono letti da `REPO/data/raw`, oppure dalla cartella indicata nella variabile d'ambiente `MGTBENCH_RAW_DIR`. Il codice verifica gli hash storici e ricostruisce gli stessi passaggi a fine frase da 90–180 parole. La coorte del pilot viene confrontata con i 42 input congelati. Non si campionano fonti in base ai punteggi dei detector.


In [ ]:
RAW_DIR = Path(os.environ.get("MGTBENCH_RAW_DIR", str(REPO / "data/raw")))
required_csv = ["Essay_LLMs.csv", "Reuters_LLMs.csv", "WP_LLMs.csv"]
missing = [name for name in required_csv if not (RAW_DIR / name).is_file()]
assert not missing, f"CSV ufficiali mancanti in {RAW_DIR}: {missing}. Impostare MGTBENCH_RAW_DIR se si usano file Kaggle Input."
COHORTS = RUN / "cohorts"
run("prepare_humanizer_pilot.py", "--raw-dir", RAW_DIR, "--output", COHORTS)
COHORT = COHORTS / (MODE + ".jsonl")
cohort = read_jsonl(COHORT)
print("Umani di calibrazione:", sum(r["split"] == "calibration" for r in cohort))
print("Fonti di test:", sum(r["split"] == "test" for r in cohort))
if MODE == "pilot":
    frozen_cohort = REPO / "inputs/pilot_42/pilot.jsonl"
    assert COHORT.read_bytes() == frozen_cohort.read_bytes(), "La coorte non coincide con i 42 input congelati."


## 4. Riscritture reali
Il preset principale usa Qwen2.5-3B con un prompt di humanization fisso, una sola generazione greedy e nessun feedback dai detector. È una baseline di rewriting via prompt, non un servizio commerciale o un modello addestrato specificamente per humanization. L'identità è il controllo tecnico.

Per DIPPER: apri un esperimento distinto, ad esempio RUN_SUFFIX="_dipper" e ACTIVE_SYSTEMS=["identity","dipper"]. Il caricamento a 4 bit è solo per il riscrittore DIPPER; il modello NLI base resta float32. Prima del download di DIPPER vengono richiesti circa 55 GiB di disco libero e una GPU adatta. Se il controllo fallisce, prosegui il primo pilot Qwen. La variante DIPPER usa un'intera finestra breve senza contesto precedente, non la procedura originale a finestre di tre frasi.


In [ ]:
REWRITES = RUN / "rewriting"
selected_args = [item for name in ACTIVE_SYSTEMS for item in ("--only", name)]
run("run_humanizers.py", "--cohort", COHORT, "--config", CONFIG,
    "--output", REWRITES, "--device", "cuda:0", *selected_args)
rewrites = read_jsonl(REWRITES / "rewrites.jsonl")
from collections import Counter
print("Esiti:", Counter((r["humanizer"], r["status"]) for r in rewrites))
for row in [r for r in rewrites if r["humanizer"] != "identity"][:2]:
    print("\nSISTEMA:", row["humanizer"], "DOMINIO:", row["domain"], "ESITO:", row["status"])
    print("ORIGINALE:", row["original"])
    print("RISCRITTURA:", row["modified"])


## 5. Detector indipendenti e calibrazione
Ogni detector vede gli stessi controlli, originali e riscritture. La soglia viene stimata sui soli umani di calibrazione per un FPR nominale del 5%; si misura poi il FPR sugli umani di test. Le decisioni sono AI se score > soglia. Il voto di maggioranza usa tre decisioni già calibrate, non la media delle probabilità grezze.

ASR è condizionata ai testi macchina riconosciuti prima della riscrittura. Le generazioni fallite restano insuccessi nel denominatore. Se nessun originale è riconosciuto, l'ASR è mancante, non zero. Nella prova su 3 testi le percentuali servono solo a controllare il funzionamento. Fakespot è eseguito nella variante a input grezzo, esplicitamente distinta dal preprocessing della sua model card.


In [ ]:
DETECTION = RUN / "detection"
run("run_detectors.py", "--cohort", COHORT, "--rewrites", REWRITES / "rewrites.jsonl",
    "--config", CONFIG, "--output", DETECTION, "--device", "cuda:0")
import pandas as pd
summary = pd.read_csv(DETECTION / "detection_summary.csv")
display(summary.loc[summary.domain == "overall", ["detector", "humanizer", "n_test_sources",
    "human_test_fpr", "original_machine_tpr", "baseline_detected", "attack_successes", "asr",
    "asr_ci95_low", "asr_ci95_high", "generations_failed"]])
display(pd.read_csv(DETECTION / "detection_macro_summary.csv"))


## 6. Stile, guardie e moduli di revisione
Le feature sono le stesse del progetto. Le scale della distanza stilometrica vengono stimate sui 300 umani di calibrazione, a livello di passaggio intero. Una distanza maggiore non significa automaticamente una riscrittura migliore. Flesch misura la leggibilità.

Le schede di revisione riguardano la conservazione semantica. Per una revisione cieca compilare i file form senza aprire prima i corrispondenti file key.


In [ ]:
STYLE = RUN / "style"
run("score_humanizer_style.py", "--cohort", COHORT,
    "--pairs", REWRITES / "pairs.jsonl", "--output", STYLE)
style = pd.read_csv(STYLE / "style_pairs.csv")
display(style.head())
print("Moduli di revisione:")
print(*[p.name for p in STYLE.glob("*form.csv")], sep="\n")


## 7. Conservazione semantica sulla GPU
Riusa MiniLM, RoBERTa-base e lo stesso DeBERTa-v3 base NLI degli esperimenti precedenti, con revisioni fissate. Ogni modello viene eseguito in un processo separato. Si mantengono documenti completi, chunk, entrambe le direzioni e aggregatori diversi. Non si applica la vecchia soglia 0,04075.

Una cella completata può essere rieseguita: gli score semanticamente compatibili vengono riusati. Se la GPU esaurisce memoria nell'NLI, imposta NLI_BATCH_SIZE=2 e riesegui questo blocco; la cache delle probabilità già prodotte è conservata.


In [ ]:
SEMANTIC = RUN / "semantic"
SCORES = SEMANTIC / "scores"
MODELS = RUN.parent / ".humanization_semantic_models"
MODELS.mkdir(parents=True, exist_ok=True)
SEMANTIC.mkdir(exist_ok=True)
source_pairs = REWRITES / "pairs.jsonl"
target_pairs = SEMANTIC / "pairs.jsonl"
if target_pairs.exists():
    assert target_pairs.read_bytes() == source_pairs.read_bytes(), "Le coppie sono cambiate: usa una directory nuova."
else:
    shutil.copy2(source_pairs, target_pairs)
assert read_jsonl(target_pairs), "Nessuna coppia valida: controlla prima le generazioni."
NLI_BATCH_SIZE = 4
pairs_hash = hashlib.sha256(target_pairs.read_bytes()).hexdigest()

def completed_stage(stem):
    meta, scores = SCORES / (stem + "_run.json"), SCORES / (stem + ".csv")
    return (meta.is_file() and scores.is_file()
            and json.loads(meta.read_text()).get("pairs_sha256") == pairs_hash)

for stage in ["lexical", "embedding", "bertscore", "nli"]:
    if completed_stage(stage):
        print("Riutilizzo:", stage)
        continue
    if stage != "lexical":
        run("download_semantic_models.py", "--model", stage, "--revision",
            config["semantic_models"][stage], "--output", MODELS)
    run("run_semantic_experiment.py", "--stage", stage, "--data-dir", SEMANTIC,
        "--output", SCORES, "--models", MODELS, "--device", "cuda:0",
        "--nli-batch-size", NLI_BATCH_SIZE, "--threads", 2)
for kind, stem in [("cosine", "aligned_cosine"), ("bertscore", "aligned_bertscore")]:
    if completed_stage(stem):
        print("Riutilizzo:", stem)
        continue
    if kind == "bertscore":
        bert_dir = MODELS / "bertscore"
        if not (bert_dir / "config.json").is_file() or not any((bert_dir / name).is_file() for name in ["model.safetensors", "pytorch_model.bin"]):
            run("download_semantic_models.py", "--model", "bertscore", "--revision",
                config["semantic_models"]["bertscore"], "--output", MODELS)
    run("score_aligned_similarity.py", "--stage", kind, "--pairs", target_pairs,
        "--alignment", SEMANTIC / "alignment.json", "--output", SCORES,
        "--model", MODELS / "bertscore", "--device", "cuda:0", "--threads", 2)


## 8. Esportazione continua e confronto dei sistemi
La suite mantiene separati evasione, conservazione e stile. I punteggi NLI non sono probabilità che il documento sia corretto. La revisione umana dei nuovi output serve a verificare quanto gli allarmi corrispondano a errori reali.


In [ ]:
score_tables = [SCORES / (stem + ".csv") for stem in [
    "lexical", "embedding", "bertscore", "nli", "aligned_cosine", "aligned_bertscore"]]
score_tables.append(STYLE / "guards.csv")
EXPORT = RUN / "continuous_suite"
score_args = [item for path in score_tables for item in ("--score", str(path))]
run("export_semantic_suite.py", "--pairs", target_pairs, *score_args, "--output", EXPORT)
continuous = pd.read_csv(EXPORT / "continuous_scores.csv")
meta = pd.DataFrame(read_jsonl(target_pairs))[["id", "domain", "generator"]]
continuous = continuous.merge(meta, on="id", validate="one_to_one")
continuous.to_csv(EXPORT / "continuous_scores_with_source_metadata.csv", index=False)
display(pd.read_csv(EXPORT / "humanizer_distributions.csv").head(20))
print("Riepilogo evasione:", DETECTION / "detection_summary.csv")
print("Risultati continui:", EXPORT / "continuous_scores_with_source_metadata.csv")
print("Stile e revisione:", STYLE)


## 9. Scarica e conserva i risultati
Lo ZIP contiene input congelati, riscritture, errori, score, soglie, configurazione e moduli di revisione. Non contiene pesi, venv o cache di download. Scaricalo prima di terminare la sessione e salva una versione privata del notebook con gli output disponibili. I risultati possono poi essere riportati nella chat per l'analisi.

Dopo lo smoke, controlla che l'identità abbia testo invariato, distanza stilometrica zero e ASR zero quando il denominatore è positivo. Poi cambia MODE in "pilot", riavvia dalla prima cella e mantieni gli stessi modelli, prompt e regole. Il passaggio a "study" riserva 126 fonti, ma va deciso dopo l'analisi del pilot.


In [ ]:
archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("Archivio da scaricare:", archive)
print("Dimensione MiB:", round(Path(archive).stat().st_size / 2**20, 2))
from IPython.display import FileLink, display
display(FileLink(archive))
